# Megaline Plan Recommendation System

## Project Overview

Megaline aims to migrate customers from legacy mobile plans to its newer offerings: **Smart** and **Ultra**. The objective of this project is to develop a machine learning classification model capable of recommending the most suitable plan based on customer usage behavior.

Using historical subscriber data, several supervised learning algorithms are evaluated to identify the model that delivers the highest predictive accuracy.

---

## Business Objective

Develop a classification model that predicts whether a customer should be assigned to the **Smart** or **Ultra** plan.

**Project Success Criterion:**  
- Accuracy ≥ 75%

---

## Dataset Description

Each observation represents one customer's monthly usage behavior.

| Feature | Description |
|----------|------------|
| `calls` | Number of calls made |
| `minutes` | Total call duration (minutes) |
| `messages` | Number of text messages sent |
| `mb_used` | Internet usage in MB |
| `is_ultra` | Target variable (Ultra = 1, Smart = 0) |

---

## 1. Data Loading and Inspection

In [98]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

In [81]:
# Global variables

random_state = 42

In [82]:
dataset_path = '/datasets/users_behavior.csv' # TripleTen platform

if not Path(dataset_path).exists():
    dataset_path = '.' + dataset_path  # Local development

In [83]:
data = pd.read_csv(dataset_path)

data.head()

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [84]:
# Check the data types of the columns
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


In [85]:
# Duplicate rows
data.duplicated().sum()

0

The dataset is clean and does not contain any missing or duplicate values. Column names are correctly formatted, and each feature has the appropriate data type for further analysis and model development.

---

## 2. Data Splitting

The dataset is divided into three subsets:

### Training Set
Used to train machine learning models.

### Validation Set
Used for hyperparameter tuning and model selection.

### Test Set
Used only for the final unbiased evaluation.

### Split Strategy

- Training Set: 60%
- Validation Set: 20%
- Test Set: 20%

In [86]:
# Features and target variable
features = data.drop(columns=['is_ultra'])
target = data['is_ultra']

# Split the dataset into training, and evaluation sets
train_features, evaluation_features, train_target, evaluation_target = train_test_split(features, target, test_size=0.4, random_state=random_state)

# Split the evaluation set into validation and test sets
validation_features, test_features, validation_target, test_target = train_test_split(evaluation_features, evaluation_target, test_size=0.5, random_state=random_state)

# Check the sizes of the datasets
print(f"Training set size: \t {train_features.shape[0]} -> {(train_features.shape[0] / data.shape[0]) * 100:.2f}%")
print(f"Validation set size: \t {validation_features.shape[0]} -> {(validation_features.shape[0] / data.shape[0]) * 100:.2f}%")
print(f"Test set size: \t \t {test_features.shape[0]} -> {(test_features.shape[0] / data.shape[0]) * 100:.2f}%")

Training set size: 	 1928 -> 59.99%
Validation set size: 	 643 -> 20.01%
Test set size: 	 	 643 -> 20.01%


## 3. Model Development

Several classification algorithms are trained and compared.

### Decision Tree Classifier

Hyperparameters explored:

- `max_depth`
- `min_samples_leaf`
- `min_samples_split`

In [87]:
# Find the best hyperparameters for the Decision Tree Classifier model
best_accuracy = 0
max_depth_range = range(1, 16)
min_samples_leaf_range = range(1, 11)
min_samples_split_range = range(2, 11)

for max_depth in max_depth_range:
    for min_samples_leaf in min_samples_leaf_range:
        for min_samples_split in min_samples_split_range:
            model = DecisionTreeClassifier(random_state=random_state, max_depth=max_depth, min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf)
            model.fit(train_features, train_target)
            accuracy = model.score(validation_features, validation_target)
            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_params = {
                    'max_depth': max_depth,
                    'min_samples_leaf': min_samples_leaf,
                    'min_samples_split': min_samples_split
                }

print(f"Best accuracy: {best_accuracy:.4f}")
print(f"Best hyperparameters: {best_params}")

Best accuracy: 0.8009
Best hyperparameters: {'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2}


### Random Forest Classifier

Hyperparameters explored:

- `n_estimators`
- `max_depth`

In [88]:
# Find the best hyperparameters for the Random Forest Classifier model
best_accuracy_rf = 0
n_estimators_range = range(10, 101, 10)
max_depth_range = range(1, 16)

for n_estimators in n_estimators_range:
    for max_depth in max_depth_range:
        model_rf = RandomForestClassifier(random_state=random_state, n_estimators=n_estimators, max_depth=max_depth)
        model_rf.fit(train_features, train_target)
        accuracy_rf = model_rf.score(validation_features, validation_target)
        if accuracy_rf > best_accuracy_rf:
            best_accuracy_rf = accuracy_rf
            best_params_rf = {
                'n_estimators': n_estimators,
                'max_depth': max_depth
            }
            
print(f"Best accuracy (Random Forest): {best_accuracy_rf:.4f}")
print(f"Best hyperparameters (Random Forest): {best_params_rf}")

Best accuracy (Random Forest): 0.8227
Best hyperparameters (Random Forest): {'n_estimators': 80, 'max_depth': 12}


### Logistic Regression

Hyperparameters explored:

- Solver selection
- Regularization settings

In [ ]:
# Find the best hyperparameters for the Logistic Regression model
best_accuracy_lr = 0
C_range = [0.001, 0.01, 0.1, 0.5, 1, 5, 10, 50, 100]
solvers = ['liblinear', 'lbfgs']

for C in C_range:
    for solver in solvers:
        model_lr = LogisticRegression(random_state=random_state, C=C, solver=solver, max_iter=2000) # class_weight = 'balanced'
        model_lr.fit(train_features, train_target)
        accuracy_lr = model_lr.score(validation_features, validation_target)
        if accuracy_lr > best_accuracy_lr:
            best_accuracy_lr = accuracy_lr
            best_params_lr = {
                'C': C,
                'solver': solver
            }

print(f"Best accuracy (Logistic Regression): {best_accuracy_lr:.4f}")
print(f"Best hyperparameters (Logistic Regression): {best_params_lr}")

Best accuracy (Logistic Regression): 0.7403
Best hyperparameters (Logistic Regression): {'C': 0.001, 'solver': 'lbfgs'}


### Model Selection Strategy

The **Decision Tree Classifier** achieved a maximum validation accuracy of `0.8009` with the following hyperparameters:


- `max_depth=10`
- `min_samples_split=2`
- `min_samples_leaf=2`

This result indicates that allowing the tree to grow to a moderate depth while keeping relatively flexible splitting criteria improved its ability to capture patterns in customer behavior.

The **Random Forest Classifier** outperformed all other models, reaching a validation accuracy of `0.8227` with:

- `n_estimators=80`
- `max_depth=12`

The ensemble approach benefited from combining multiple decision trees, reducing overfitting and improving generalization. Increasing the number of trees and allowing a deeper structure enabled the model to capture more complex relationships within the data.

The **Logistic Regression** model obtained a maximum accuracy of `0.7403` using:

- `C=0.001`
- `solver='lbfgs'`

Although Logistic Regression provided a simple and interpretable baseline, its performance was lower than both tree-based models and did not exceed the target accuracy threshold. This suggests that the relationship between the features and the target variable is likely non-linear, making tree-based algorithms better suited for this classification task.

Among the evaluated models, **Random Forest** delivered the best performance with an accuracy of `0.8227`, exceeding the project's required threshold of 0.75. Therefore, it was selected as the final model for predicting whether a customer should be assigned to the Smart or Ultra plan based on usage behavior.

---

## 4. Model Evaluation

The Random Forest model with the highest validation accuracy was chosen as the final model. After training, its performance was evaluated on the test dataset to determine how well it generalizes to unseen data.


In [97]:
# Create the final Random Forest Classifier model with the best hyperparameters
final_model = RandomForestClassifier(random_state=random_state, n_estimators=80, max_depth=12)

# Train the final model on the training set
final_model.fit(train_features, train_target)

# Evaluate the final model on the test set
test_accuracy = final_model.score(test_features, test_target)

print(f"Test accuracy (Random Forest): {test_accuracy:.4f}")

Test accuracy (Random Forest): 0.8196


The Random Forest model achieved a test accuracy of `0.8196` (**81.96%**), surpassing the required threshold of 0.75. These results suggest that the model can reliably classify users into the appropriate Megaline plan and generalizes well to new data.

---

## 5. Sanity Check

To establish a baseline for comparison, a Dummy Classifier was created and evaluated using the test dataset. This model does not learn meaningful patterns from the data and serves as a reference point for measuring the effectiveness of more advanced machine learning models.

In [99]:
# Create a dummy classifier as a baseline model
dummy_model = DummyClassifier(strategy='most_frequent', random_state=random_state)
dummy_model.fit(train_features, train_target)
dummy_accuracy = dummy_model.score(test_features, test_target)
print(f"Test accuracy (Dummy): {dummy_accuracy:.4f}")

Test accuracy (Dummy): 0.6967


The Dummy Classifier achieved a test accuracy of `0.6967` (69.67%).

Since the Random Forest model achieved a higher accuracy than the Dummy Classifier, it demonstrates that the model successfully learned relevant patterns from customer behavior rather than relying on simple baseline predictions. Therefore, the Random Forest model provides meaningful predictive value and can be considered a better solution for recommending the appropriate mobile plan.

## Final Conclusions

The objective of this project was to develop a machine learning model capable of recommending the most suitable Megaline mobile plan (Smart or Ultra) based on customer behavior data. To achieve this, the dataset was divided into training, validation, and test sets, allowing for proper model training, hyperparameter tuning, and final evaluation.

Several classification algorithms were explored and compared, including Decision Tree, Random Forest, and Logistic Regression models. Hyperparameters were adjusted to improve predictive performance and identify the most effective approach. Among all evaluated models, the Random Forest Classifier achieved the highest accuracy and demonstrated the best overall performance on the validation and test datasets.

To further assess the model's effectiveness, a Dummy Classifier was used as a baseline. The dummy model achieved a test accuracy of 0.6967, while the Random Forest model obtained a higher accuracy score, confirming that it successfully learned meaningful patterns from customer behavior rather than relying on simple or random predictions.

Overall, the project successfully met the required accuracy threshold of 0.75 and demonstrated that machine learning can be effectively used to recommend mobile plans based on user activity. The results indicate that the Random Forest model is a reliable solution for supporting Megaline's plan recommendation process and can help guide customers toward plans that better match their usage patterns.